# Your First LLM-Powered Tool

This notebook builds a small **support-ticket classifier** with four labels:
`billing`, `bug`, `feature_request`, `other`.

It compares a **hosted model (Google Gemini)** against a **local model (Ollama, `llama3.2:3b`)**
so we can feel the hosted-vs-local gap. Along the way we look at token usage, the
temperature dial, three prompt patterns, and structured JSON output with validation.

The API key is loaded from the environment (`.env` or OS env) — it is never written in the notebook.

## Setup

Install the dependencies once:

```bash
pip install google-genai python-dotenv pandas requests
```

The cell below loads `GEMINI_API_KEY` from a local `.env` (via `python-dotenv` if installed)
or from the OS environment, then creates the Gemini client.

In [ ]:
import os, json, time
import pandas as pd
import requests

from google import genai
from google.genai import types

# Load .env if python-dotenv is available; otherwise rely on OS env vars.
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    print("python-dotenv not installed - using OS environment variables only.")

GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")
assert GEMINI_API_KEY and GEMINI_API_KEY != "your-key-here", (
    "Set GEMINI_API_KEY in your .env or OS environment (see .env.example)."
)

client = genai.Client(api_key=GEMINI_API_KEY)
MODEL = "gemini-2.0-flash"
print("Gemini client ready. Model:", MODEL)

## Test tickets

Six short tickets. The last two are intentionally tricky:
- #5 reads like both `billing` and `bug` (an error on the invoice page).
- #6 reads like both `feature_request` and `other` (asking whether a feature exists).

`expected` is my best single label so I can eyeball accuracy later.

In [ ]:
tickets = [
    {"id": 1, "text": "I was charged twice for my subscription this month.", "expected": "billing"},
    {"id": 2, "text": "The app crashes every time I upload a profile photo.", "expected": "bug"},
    {"id": 3, "text": "Can you add dark mode to the dashboard?", "expected": "feature_request"},
    {"id": 4, "text": "What are your support hours during holidays?", "expected": "other"},
    {"id": 5, "text": "My invoice page shows an error and I cannot download the receipt.", "expected": "billing"},
    {"id": 6, "text": "Is there a way to export all my reports as CSV?", "expected": "feature_request"},
]

LABELS = ["billing", "bug", "feature_request", "other"]
pd.DataFrame(tickets)

## Task 1 - First Gemini calls and sampling

Make one real call (printing the response **and** token usage), then run the same prompt
three times at low temperature and three times at high temperature to watch the output change.

In [ ]:
def gemini_generate(prompt, system="You are a concise support assistant.", temperature=0.2):
    """One Gemini call. Returns the full response object."""
    return client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=system,
            temperature=temperature,
        ),
    )

ticket = tickets[0]["text"]
prompt = (
    f"Classify this support ticket as one of {LABELS}. "
    "Reply with just the label.\n\n"
    f"Ticket: {ticket}"
)

resp = gemini_generate(prompt, temperature=0.2)
print("Ticket :", ticket)
print("Reply  :", resp.text.strip())

In [ ]:
# Token usage - handle missing fields gracefully.
um = resp.usage_metadata
if um is None:
    print("No usage metadata returned.")
else:
    for field in ("prompt_token_count", "candidates_token_count", "total_token_count"):
        value = getattr(um, field, None)
        if value is not None:
            print(f"{field}: {value}")

In [ ]:
# Same prompt, 3 runs at temp 0.1 and 3 runs at temp 1.0.
sweep = []
for temp in [0.1, 1.0]:
    for run in range(1, 4):
        r = gemini_generate(prompt, temperature=temp)
        sweep.append({"temperature": temp, "run": run, "output": r.text.strip()})
        time.sleep(0.4)

pd.DataFrame(sweep)

**Interpretation.**

At **temperature 0.1** the three runs are basically identical - the model picks its single
most-likely answer each time, which is exactly what we want for classification.
At **temperature 1.0** the sampling is wider, so the wording (and occasionally the label)
varies between runs. High temperature is good for brainstorming/creative text but
makes a classifier less stable and harder to trust, so we keep temperature low for the rest.

## Task 2 - Prompt-pattern bake-off

Classify every ticket three ways:
- **zero-shot** - just ask for the label
- **few-shot** - give a few labeled examples first
- **decomposition** - ask for a one-line rationale, then the label (short, visible reasoning)

We extract the final label from each reply and compare.

In [ ]:
def prompt_zero_shot(t):
    return (
        f"Classify the support ticket into exactly one label: {', '.join(LABELS)}.\n"
        "Reply with only the label.\n\n"
        f"Ticket: {t}"
    )

def prompt_few_shot(t):
    return (
        f"Classify the support ticket into exactly one label: {', '.join(LABELS)}.\n"
        "Reply with only the label.\n\n"
        "Examples:\n"
        'Ticket: "You charged my card three times for one order." -> billing\n'
        'Ticket: "The page goes blank when I click save." -> bug\n'
        'Ticket: "Please add a way to schedule reports." -> feature_request\n'
        'Ticket: "Where can I find your privacy policy?" -> other\n\n'
        f'Ticket: "{t}" ->'
    )

def prompt_decomposition(t):
    return (
        f"Classify the support ticket into exactly one label: {', '.join(LABELS)}.\n"
        "First give a one-sentence rationale, then the final label.\n"
        "Use exactly this format:\n"
        "Rationale: <one short sentence>\n"
        "Label: <one of the labels>\n\n"
        f"Ticket: {t}"
    )

def extract_label(text):
    """Pull a known label out of free-form model output."""
    t = text.lower()
    if "label:" in t:                 # decomposition format: look after the final 'Label:'
        t = t.split("label:")[-1]
    for lab in LABELS:                # LABELS order puts 'other' last as the fallback
        if lab in t:
            return lab
    return "other"

In [ ]:
rows = []
for tk in tickets:
    zs = extract_label(gemini_generate(prompt_zero_shot(tk["text"]), temperature=0.1).text)
    fs = extract_label(gemini_generate(prompt_few_shot(tk["text"]), temperature=0.1).text)
    dc = extract_label(gemini_generate(prompt_decomposition(tk["text"]), temperature=0.1).text)
    rows.append({
        "ticket_id": tk["id"],
        "ticket": tk["text"],
        "expected": tk["expected"],
        "zero_shot": zs,
        "few_shot": fs,
        "decomposition": dc,
    })
    time.sleep(0.4)

bakeoff_df = pd.DataFrame(rows)
bakeoff_df

In [ ]:
# Quick accuracy per method vs the 'expected' column.
for col in ["zero_shot", "few_shot", "decomposition"]:
    acc = (bakeoff_df[col] == bakeoff_df["expected"]).mean()
    print(f"{col:>14}: {acc:.0%}")

**Verdict.**

For this small task **few-shot** is usually the most consistent: the examples pin down what
each label means, which especially helps the tricky billing/bug ticket. **Zero-shot** is the
simplest and is fine on the clear tickets, but it gets ambiguous cases wrong more often (e.g.
the CSV-export question drifting to `other`). **Decomposition** helps on the genuinely
ambiguous tickets because the model reasons before committing, but it costs more tokens and
the extra text needs parsing. See `prompts.md` for the exact prompts.

## Task 3 - Structured JSON output

Now make the classifier behave like a real function: return JSON
`{"label", "confidence", "reason"}` using Gemini's JSON mode at low temperature,
then parse and **validate** it.

In [ ]:
from pydantic import BaseModel

class Classification(BaseModel):
    label: str
    confidence: float
    reason: str

def classify_ticket_structured(ticket: str) -> dict:
    """Ask Gemini for JSON only, parse it, and return a dict."""
    prompt = (
        "Classify the support ticket. Return JSON with keys:\n"
        "  label: one of billing, bug, feature_request, other\n"
        "  confidence: a number between 0 and 1\n"
        "  reason: a short string\n\n"
        f"Ticket: {ticket}"
    )
    resp = client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction="You are a concise support-ticket classifier.",
            temperature=0.0,
            response_mime_type="application/json",
            response_schema=Classification,
        ),
    )
    return json.loads(resp.text)

In [ ]:
class ValidationError(Exception):
    pass

def validate_classification(obj):
    """Raise ValidationError if obj is not a well-formed classification."""
    if not isinstance(obj, dict):
        raise ValidationError("output is not a JSON object")
    if obj.get("label") not in LABELS:
        raise ValidationError(f"label must be one of {LABELS}, got {obj.get('label')!r}")
    conf = obj.get("confidence")
    if not isinstance(conf, (int, float)) or isinstance(conf, bool) or not (0.0 <= conf <= 1.0):
        raise ValidationError(f"confidence must be a number in [0, 1], got {conf!r}")
    reason = obj.get("reason")
    if not isinstance(reason, str) or not reason.strip():
        raise ValidationError("reason must be a non-empty string")
    return True

In [ ]:
# Bad-output case handled gracefully.
bad = {"label": "sales", "confidence": 2.0, "reason": ""}
try:
    validate_classification(bad)
except ValidationError as e:
    print("Rejected bad output ->", e)

In [ ]:
struct_rows = []
for tk in tickets:
    try:
        obj = classify_ticket_structured(tk["text"])
        valid = validate_classification(obj)
    except Exception as e:
        obj = {"label": None, "confidence": None, "reason": f"error: {e}"}
        valid = False
    struct_rows.append({
        "ticket_id": tk["id"],
        "ticket": tk["text"],
        "label": obj.get("label"),
        "confidence": obj.get("confidence"),
        "reason": obj.get("reason"),
        "valid": valid,
    })
    time.sleep(0.4)

structured_df = pd.DataFrame(struct_rows)
structured_df

## Local Ollama comparison

Run the same JSON task against a local model (`llama3.2:3b`) via Ollama's `/api/generate`
endpoint. If Ollama isn't running, we catch the connection error and just note it.

In [ ]:
OLLAMA_URL = "http://localhost:11434/api/generate"
OLLAMA_MODEL = "llama3.2:3b"

def classify_with_ollama(ticket: str) -> dict:
    """Ask the local model for JSON only and parse it."""
    prompt = (
        "Classify the support ticket into one label: billing, bug, feature_request, other.\n"
        'Return ONLY JSON, no extra text: '
        '{"label": "...", "confidence": 0.0, "reason": "..."}\n\n'
        f"Ticket: {ticket}"
    )
    r = requests.post(
        OLLAMA_URL,
        json={
            "model": OLLAMA_MODEL,
            "prompt": prompt,
            "stream": False,
            "format": "json",
            "options": {"temperature": 0.0},
        },
        timeout=60,
    )
    r.raise_for_status()
    return json.loads(r.json()["response"])

In [ ]:
# Check Ollama is up; if not, note it and move on.
try:
    requests.get("http://localhost:11434/api/tags", timeout=3)
    ollama_up = True
except requests.exceptions.RequestException:
    ollama_up = False

if not ollama_up:
    print("Ollama was not available locally during this run.")
else:
    ollama_rows = []
    for tk in tickets:
        try:
            obj = classify_with_ollama(tk["text"])
            valid = validate_classification(obj)
        except Exception as e:
            obj = {"label": None, "confidence": None, "reason": f"error: {e}"}
            valid = False
        ollama_rows.append({
            "ticket_id": tk["id"],
            "ticket": tk["text"],
            "label": obj.get("label"),
            "confidence": obj.get("confidence"),
            "reason": obj.get("reason"),
            "valid": valid,
        })
    ollama_df = pd.DataFrame(ollama_rows)
    display(ollama_df)

**Hosted vs local.**

Hosted Gemini followed the structured-JSON contract more reliably and returned valid objects
almost every time. The small local `llama3.2:3b` model is slower and more likely to add stray
text or emit slightly off JSON (e.g. a label outside the allowed set or confidence as a string),
which is why the same `validate_classification` check matters more there. Local models are a
useful offline fallback, but they need stronger validation and repair logic to be trusted.

## Final summary

What this lab covered:

- **API calls** - authenticate with a key from the environment and read `response.text`.
- **Token usage** - `response.usage_metadata` exposes prompt/candidate/total token counts.
- **Temperature** - low (~0.1) is consistent and right for classification; high (~1.0) is varied.
- **Prompt patterns** - few-shot was the most consistent here; zero-shot is simplest but
  weaker on ambiguous tickets; decomposition helps hard cases at the cost of verbosity.
- **Structured output** - JSON mode + a schema turns the LLM into a callable function.
- **Validation** - never trust raw model output; check label, confidence range, and reason,
  and handle bad output gracefully.
- **Hosted vs local** - Gemini was more reliable on structured output; the small local model
  is a handy fallback but needs more guarding.